In [ ]:
!pip install -q -U bitsandbytes

import os
os.environ['HF_HOME'] = '/kaggle/temp/hf'
os.makedirs('/kaggle/temp/hf', exist_ok=True)

import glob, re, json, time, gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct'
BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
asr_dirs = glob.glob('/kaggle/input/**/asr_raw', recursive=True)
assert len(asr_dirs) == 1, asr_dirs
ASR_DIR = asr_dirs[0]
WORK = '/kaggle/working'

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
uids = ['train/' + f for f in train.filename] + ['test/' + f for f in test.filename]
assert len(set(uids)) == len(uids) == 985

texts = {}
for u in uids:
    with open(f'{ASR_DIR}/{u.replace("/", "__")}.json') as f:
        r = json.load(f)
    texts[u] = re.sub(r'\s+', ' ', ' '.join(s['text'].strip() for s in r['segments'])).strip()
print('transcripts loaded:', len(texts), '| empty:', sum(1 for t in texts.values() if not t))
assert torch.cuda.is_available(), 'GPU is off'

tok = AutoTokenizer.from_pretrained(MODEL_ID)
assert tok.is_fast, 'need a fast tokenizer for offset mapping'
kw = dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map='auto')
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, **kw)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, **kw)
model.eval()
NL, HD = model.config.num_hidden_layers, model.config.hidden_size
print('loaded | layers:', NL, '| hidden:', HD)
for g in range(torch.cuda.device_count()):
    print('gpu', g, 'memory GB:', round(torch.cuda.memory_allocated(g) / 1e9, 1))

In [ ]:
# M2: prompt, function, smoke test
from scipy.stats import spearmanr

SYSTEM = 'You are an expert examiner of spoken English grammar.'
MARK = 'Transcript:\n"""\n'
TAIL = 'Answer with a single digit from 1 to 5 and nothing else.'
INTRO = ('Below is an automatic speech-recognition transcript of a 45 to 60 second spoken response. '
         'The transcript comes from speech recognition, so ignore capitalization and punctuation; '
         'repetitions and false starts are real features of the speech.\n\n')
RUBRIC = """1: The speech struggles with proper sentence structure and syntax, with limited control over simple grammatical structures and memorized sentence patterns.
2: Limited understanding of sentence structure and syntax. Simple structures are used, but basic sentence-structure and grammatical mistakes are consistent. Sentences may be left incomplete.
3: Decent grasp of sentence structure but errors in grammatical structure, or decent grasp of grammatical structure but errors in sentence syntax and structure.
4: Strong understanding of sentence structure and syntax, with consistently good control of grammar. Occasional errors are generally minor, do not cause misunderstanding, and most can be self-corrected.
5: High grammatical accuracy and adept control of complex grammar. Grammar is used accurately and effectively, with seldom any noticeable mistakes. Complex structures are handled well and the speaker corrects themselves when necessary."""
TASK = ("Rate ONLY the speaker's grammar (sentence structure, syntax, grammatical accuracy and control of complex structures). "
        'Do not rate vocabulary, pronunciation, fluency or the quality of the ideas.\n\nGrammar score rubric:\n' + RUBRIC)

def build(text):
    user = INTRO + TASK + '\n\n' + MARK + text + '\n"""\n\n' + TAIL
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

digit_ids = []
for d in '12345':
    ids = tok.encode(d, add_special_tokens=False)
    assert len(ids) == 1, (d, ids)
    digit_ids.append(ids[0])

# same relative depths as the 7B run (layers 6, 9, 12, 16, 20, 24 of 28)
POOL_LAYERS = [int(round(f * NL)) for f in (0.21, 0.32, 0.43, 0.57, 0.71, 0.86)]
print('pooled layers:', POOL_LAYERS)

@torch.inference_mode()
def run(text):
    prompt = build(text)
    enc = tok(prompt, return_tensors='pt', return_offsets_mapping=True)
    offs = enc.pop('offset_mapping')[0]
    enc = enc.to(model.device)
    out = model(**enc, output_hidden_states=True, use_cache=False)
    logits = out.logits[0, -1].float()
    lp = torch.log_softmax(logits, -1)
    five = logits[digit_ids].cpu().numpy()
    p = np.exp(five - five.max())
    p /= p.sum()
    a = prompt.rfind(MARK) + len(MARK)
    b = a + len(text)
    assert prompt[a:b] == text, 'transcript span not found'
    m = (offs[:, 0] >= a) & (offs[:, 1] <= b) & (offs[:, 1] > offs[:, 0])
    idx = torch.nonzero(m).squeeze(-1).to(model.device)
    pool = torch.stack([out.hidden_states[L][0].index_select(0, idx).float().mean(0)
                        for L in POOL_LAYERS], 0).cpu().numpy()
    return dict(e=float((p * np.arange(1, 6)).sum()), ent=float(-(p * np.log(p + 1e-12)).sum()),
                mass=float(lp[digit_ids].exp().sum().item()), p=p.tolist(), ntr=int(idx.numel()), pool=pool)

lab = train.set_index('filename').label
samp = [u for u in uids[:len(train)] if lab[u.split('/')[1]] > 0][:40]
t0 = time.time()
res = [run(texts[u]) for u in samp]
sec = (time.time() - t0) / len(samp)
yy = np.array([lab[u.split('/')[1]] for u in samp])
e = np.array([r['e'] for r in res])
print('seconds per file:', round(sec, 2), '| estimated total minutes:', round(sec * 985 / 60, 1))
print('transcript tokens: min', min(r['ntr'] for r in res), 'max', max(r['ntr'] for r in res))
print('pooled shape:', res[0]['pool'].shape, '(expect (6, 5120))')
print('NaN present:', not all(np.isfinite(r['pool']).all() and np.isfinite(r['e']) for r in res))
print('largest abs pooled value:', round(float(max(np.abs(r['pool']).max() for r in res)), 1))
print('digit mass min:', round(min(r['mass'] for r in res), 3), '| Spearman with label (n=40, very noisy):', round(spearmanr(e, yy)[0], 2))
for g in range(torch.cuda.device_count()):
    print('gpu', g, 'peak memory GB:', round(torch.cuda.max_memory_allocated(g) / 1e9, 1))

In [ ]:
N = len(uids)
P_POOL, P_REC = f'{WORK}/_part_pool.npy', f'{WORK}/_part_rec.json'

if os.path.exists(P_REC):
    pool_all, recs = np.load(P_POOL), json.load(open(P_REC))
    start = len(recs)
    print('resuming from', start)
else:
    pool_all, recs, start = np.zeros((N, len(POOL_LAYERS), HD), dtype=np.float16), [], 0

n_clipped = 0
t0 = time.time()
for i in range(start, N):
    r = run(texts[uids[i]])
    n_clipped += int((np.abs(r['pool']) > 6e4).sum())
    pool_all[i] = np.clip(r['pool'], -6e4, 6e4).astype(np.float16)
    row = {'uid': uids[i], 'ntr': r['ntr'], 'e': r['e'], 'ent': r['ent'], 'mass': r['mass']}
    for k in range(5):
        row[f'p{k + 1}'] = r['p'][k]
    recs.append(row)
    if (i + 1) % 100 == 0 or i + 1 == N:
        np.save(P_POOL, pool_all)
        json.dump(recs, open(P_REC, 'w'))
        el = time.time() - t0
        print(f'{i + 1}/{N} | {el / 60:.1f} min | est remaining {(N - i - 1) * el / (i + 1 - start) / 60:.1f} min')

print('pooled values clipped at +-60000:', n_clipped)
print('NaN or inf in pooled states:', not np.isfinite(pool_all).all())
S = pd.DataFrame(recs)
print('rows:', len(S), '| NaN in scores:', int(S.drop(columns='uid').isna().sum().sum()))
S.to_csv(f'{WORK}/llm14_scores.csv', index=False)
np.save(f'{WORK}/llm14_meanpool.npy', pool_all)
pd.Series(uids).to_csv(f'{WORK}/llm14_uids.csv', index=False, header=['uid'])
for p in (P_POOL, P_REC):
    os.remove(p)
print('saved', pool_all.shape, '|', sorted(os.listdir(WORK)))

In [ ]:
prev = pd.read_csv(glob.glob('/kaggle/input/**/llm_views.csv', recursive=True)[0]).set_index('uid')
S = pd.read_csv(f'{WORK}/llm14_scores.csv')
print('corr of 14B and 7B rubric expected scores (all 985 files):',
      round(float(np.corrcoef(S.e, prev.loc[S.uid, 'rubric_e'])[0, 1]), 3))

dur = {}
for u in uids:
    with open(f'{ASR_DIR}/{u.replace("/", "__")}.json') as f:
        dur[u] = json.load(f)['duration']
S['label'] = [lab[u.split('/')[1]] if u.startswith('train/') else np.nan for u in S.uid]
S['dg'] = np.where(S.uid.map(dur) < 40, 'short', np.where(S.uid.map(dur) < 52, 'dur_45s', 'dur_60s'))
trn = S[S.label > 0]
print('Spearman(14B expected score, label), train non-zero: all', round(spearmanr(trn.e, trn.label)[0], 3),
      '| 45s', round(spearmanr(trn[trn.dg == 'dur_45s'].e, trn[trn.dg == 'dur_45s'].label)[0], 3),
      '| 60s', round(spearmanr(trn[trn.dg == 'dur_60s'].e, trn[trn.dg == 'dur_60s'].label)[0], 3))
print('reference, 7B rubric view: all 0.622 | 45s 0.657 | 60s 0.627')